# CVD Analytics

Add coordinated vulnerability disclosure analytics: timelines, flags, inference.

In [ ]:
from vulnflow import VulnDataset
from vulnflow.transforms.flatten import CVSSFlattenTransform, CWEPrimaryTransform, CPEPrimaryTransform
from vulnflow.transforms.computed import CVDStateTransform
from vulnflow.transforms.inferences import EventInference, TagInference
import tempfile
from pathlib import Path

## Load with CVD transforms

Progress bar tracks all transforms.

In [ ]:
ds = VulnDataset.from_nvd("../data/nvdcve-2.0-Modified.json").transform([
    CVSSFlattenTransform(),
    CWEPrimaryTransform(), 
    CPEPrimaryTransform(),
    EventInference(),
    CVDStateTransform(),
    TagInference(),
])

df = ds.to_pandas()

## CVD timeline columns

EventInference adds timestamps + confidence scores for V/F/D/P/X/A events.

In [ ]:
[c for c in df.columns if c.endswith('_timestamp') or c.endswith('_confidence')]

## CVD analytics

CVDStateTransform adds 19 computed fields: state, flags, lags.

In [ ]:
[c for c in df.columns if c.startswith('cvd_') or c.startswith('is_') or c.endswith('_lag_days')]

## Zero-day exploits

In [ ]:
zero_days = df[df['is_zero_day_exploit'] == True]
zero_days[['cve_id', 'cvss_base_score', 'exploit_lag_days', 'description']].head(10)

## Fix lag distribution

In [ ]:
df['fix_lag_days'].describe()

## CVD state codes

6-character state showing event progression (e.g., "vfd---" = vendor aware + fixed + disclosed).

In [ ]:
df['cvd_state'].value_counts().head(10)

## Cache enriched data

Save once, reload instantly with all transforms.

In [ ]:
demo_dir = Path(tempfile.mkdtemp())
enriched_path = demo_dir / "enriched.parquet"

ds.to_parquet(enriched_path)

In [ ]:
# Instant reload
df_cached = VulnDataset.from_parquet(enriched_path).to_pandas()
len(df_cached)

## Production pattern with incremental

```python
transforms = [
    CVSSFlattenTransform(),
    CWEPrimaryTransform(),
    CPEPrimaryTransform(),
    EventInference(),
    CVDStateTransform(),
]

# ONE-TIME: Build enriched cache
VulnDataset.from_nvd("nvdcve-2.0-Recent.json") \
    .transform(transforms) \
    .to_parquet("cache_enriched.parquet")

# EVERY SESSION: Process delta with same transforms
VulnDataset.from_nvd("nvdcve-2.0-Modified.json") \
    .transform(transforms) \
    .to_parquet("delta_enriched.parquet")

# Merge and analyze
df = VulnDataset.from_parquet([
    "cache_enriched.parquet",
    "delta_enriched.parquet"
]).to_pandas()
```

In [ ]:
# Cleanup
import shutil
shutil.rmtree(demo_dir)